# Train the multimodal dementia model on Colab

**Workflow:** train here (GPU) → download `best_model.pt` → run inference locally.

First: **Runtime → Change runtime type → GPU (T4 is fine)**.

Steps below: (1) get the code, (2) install deps, (3) smoke test, (4) train, (5) download the checkpoint.

## 1. Get the code
Either clone your GitHub repo (recommended) **or** upload the project folder. Edit the URL below.

In [ ]:
# Option A: clone from GitHub (replace with your repo URL)
REPO_URL = "https://github.com/<your-username>/Neuralyzer.git"
!git clone $REPO_URL
%cd Neuralyzer

# Option B (instead of cloning): use the Files panel to upload the project,
# then `%cd` into it. Comment out Option A if you upload manually.

## 2. Install training dependencies
Colab already has a CUDA `torch`/`torchaudio`. We install everything **except** torch to avoid breaking the pre-installed CUDA build.

In [ ]:
!pip install -q transformers>=4.40.0 scikit-learn>=1.3.0 PyYAML>=6.0 tqdm>=4.65.0 soundfile>=0.12.1

import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## 3. Smoke test (mock data)
Confirms the full pipeline runs and loss decreases. Downloads HuBERT + BERT on first run.

In [ ]:
!python scripts/smoke_test.py

## 4. Train

**Mock data (pipeline check):** metrics are meaningless — this only proves the training loop works.

```
!python train.py --config src/configs/default.yaml --dataset mock --single-seed --max-epochs 3
```

**Real data (ADReSS / PROCESS-2):** upload the dataset, implement the matching loader in `src/datasets/`, then set `--dataset` and `--data-root`. Full protocol = all 5 seeds:

```
!python train.py --config src/configs/default.yaml --dataset adress --data-root /content/adress
```

If you hit CUDA OOM: lower `--max-epochs` for testing, or reduce `train.batch_size` in the YAML.

In [ ]:
!python train.py --config src/configs/default.yaml --dataset mock --single-seed --max-epochs 3

## 5. Download the trained checkpoint
`train.py` copies the best run to `runs/best_model.pt` (weights + config bundled). Download it and use it locally with `inference.py`.

In [ ]:
from google.colab import files
import os

ckpt = "runs/best_model.pt"
assert os.path.exists(ckpt), f"{ckpt} not found — did training finish?"
print("size (MB):", round(os.path.getsize(ckpt) / 1e6, 1))
files.download(ckpt)

### (Optional) Save to Google Drive instead
```python
from google.colab import drive
drive.mount('/content/drive')
import shutil
shutil.copy('runs/best_model.pt', '/content/drive/MyDrive/best_model.pt')
```